# analysis_manifest_batch_backfill_v1

目标：验证 `src.analysis.scan_analysis_cards` 和 `backfill_legacy_analysis_cards` 可以先审计旧 research card，再只对可迁移 card 补 `analysis_manifest.json`。

约束：不批量改真实历史研究目录；本 notebook 使用隔离的模拟 card，证明框架入口和 guardrail。

In [1]:
from pathlib import Path
import json
import sys
import pandas as pd

REPO_ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'AGENTS.md').exists() and (path / 'src').exists())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.analysis import backfill_analysis_manifest, backfill_legacy_analysis_cards, scan_analysis_cards

CARD = REPO_ROOT / 'research/analysis_manifest_batch_backfill_v1'
SANDBOX = CARD / 'sandbox_cards'
SANDBOX.mkdir(parents=True, exist_ok=True)
{'card': str(CARD.relative_to(REPO_ROOT)), 'sandbox': str(SANDBOX.relative_to(REPO_ROOT))}

{'card': 'research/analysis_manifest_batch_backfill_v1',
 'sandbox': 'research/analysis_manifest_batch_backfill_v1/sandbox_cards'}

## 构造旧 card

这里构造三类目录：缺少 manifest 但有公开 analysis sidecar 的旧 card、已经有 manifest 的 card、只有预测明细的私有目录。

In [2]:
def write_legacy_card(path: Path) -> None:
    path.mkdir(parents=True, exist_ok=True)
    (path / 'analysis_summary.json').write_text(json.dumps({
        'schema_version': 'analysis.report.v1',
        'status': 'legacy_completed',
        'start': '2026-01-01',
        'end': '2026-06-09',
        'model_count': 2,
    }, ensure_ascii=False, indent=2), encoding='utf-8')
    pd.DataFrame([
        {'rank': 1, 'model': 'ridge_mse', 'return_pct': 3.1, 'stability_score': 2.0},
        {'rank': 2, 'model': 'gcn_mse', 'return_pct': 2.4, 'stability_score': 1.2},
    ]).to_csv(path / 'model_comparison.csv', index=False)
    pd.DataFrame([
        {'model': 'ridge_mse', 'regime': 'bull', 'return_pct': 4.0},
        {'model': 'ridge_mse', 'regime': 'stress', 'return_pct': -1.2},
    ]).to_csv(path / 'regime_metrics.csv', index=False)
    pd.DataFrame([{'model': 'ridge_mse', 'window': 'test_2026', 'return_pct': 3.1}]).to_csv(path / 'window_metrics.csv', index=False)
    pd.DataFrame([{'model': 'ridge_mse', 'segment_type': 'all', 'segment': 'all', 'diagnosis': '基准模型。'}]).to_csv(path / 'model_effect_deltas.csv', index=False)
    pd.DataFrame([{'model': 'ridge_mse', 'prediction': 0.1}]).to_csv(path / 'predictions.csv', index=False)
    pd.DataFrame([{'model': 'ridge_mse', 'nav': 1.0}]).to_csv(path / 'daily_backtest.csv', index=False)

legacy = SANDBOX / 'legacy_card'
manifested = SANDBOX / 'manifested_card'
private_only = SANDBOX / 'private_only'
write_legacy_card(legacy)
write_legacy_card(manifested)
backfill_analysis_manifest(
    manifested,
    dataset_name='already_manifested',
    universe=('SOXL',),
    calendar_path='data/qlib/soxl/calendars/day.txt',
    data_paths=(str(manifested),),
    overwrite=True,
)
private_only.mkdir(exist_ok=True)
pd.DataFrame([{'model': 'ridge_mse', 'prediction': 0.1}]).to_csv(private_only / 'predictions.csv', index=False)
sorted(p.name for p in SANDBOX.iterdir())

['legacy_card', 'manifested_card', 'private_only']

## 扫描但不修改

扫描表用于 review 和 schedule/dashbord 上游选择。它暴露可迁移状态，但不把预测明细当作公开 dashboard artifact。

In [3]:
scan = scan_analysis_cards(SANDBOX)
scan.to_csv(CARD / 'card_scan.csv', index=False)
scan

,card_path,relative_card_path,card_name,status,migration_action,has_manifest,has_summary,public_sidecar_count,private_sidecar_count,public_artifact_keys,private_artifact_keys
0,/Users/rziiiii/dev/workspace/research/analysis...,legacy_card,legacy_card,backfillable,backfill_analysis_manifest,False,True,5,2,"model_comparison,model_effect_deltas,regime_me...","daily_backtest,predictions"
1,/Users/rziiiii/dev/workspace/research/analysis...,manifested_card,manifested_card,already_manifested,keep_manifest,True,True,5,2,"model_comparison,model_effect_deltas,regime_me...","daily_backtest,predictions"
2,/Users/rziiiii/dev/workspace/research/analysis...,private_only,private_only,not_analysis_card,ignore,False,False,0,1,,predictions


In [4]:
assert set(scan['status']) == {'backfillable', 'already_manifested', 'not_analysis_card'}
legacy_row = scan.loc[scan['card_name'] == 'legacy_card'].iloc[0]
assert legacy_row['migration_action'] == 'backfill_analysis_manifest'
assert 'predictions' not in legacy_row['public_artifact_keys']
assert 'daily_backtest' not in legacy_row['public_artifact_keys']
scan[['card_name', 'status', 'migration_action', 'public_sidecar_count', 'private_sidecar_count']]

,card_name,status,migration_action,public_sidecar_count,private_sidecar_count
0,legacy_card,backfillable,backfill_analysis_manifest,5,2
1,manifested_card,already_manifested,keep_manifest,5,2
2,private_only,not_analysis_card,ignore,0,1


## 批量回填

批量入口只处理 `backfillable` card；已有 manifest 保持不动，私有明细目录跳过。

In [5]:
batch = backfill_legacy_analysis_cards(
    SANDBOX,
    dataset_name='batch_backfill_demo',
    universe=('SOXL', 'SOXX'),
    calendar_path='data/qlib/soxl/calendars/day.txt',
)
(CARD / 'batch_backfill_summary.json').write_text(json.dumps(batch, ensure_ascii=False, indent=2), encoding='utf-8')
batch

{'schema_version': 'analysis.manifest_backfill_batch.v1',
 'root_dir': '/Users/rziiiii/dev/workspace/research/analysis_manifest_batch_backfill_v1/sandbox_cards',
 'backfilled': 1,
 'skipped': 2,
 'cards': [{'card_name': 'legacy_card',
   'card_path': '/Users/rziiiii/dev/workspace/research/analysis_manifest_batch_backfill_v1/sandbox_cards/legacy_card',
   'status': 'backfillable',
   'action': 'backfilled',
   'manifest_hash': 'd240f234b0472748b6a3fc0f22641a7ef595deaf34291881f1738db417e43d93'},
  {'card_name': 'manifested_card',
   'card_path': '/Users/rziiiii/dev/workspace/research/analysis_manifest_batch_backfill_v1/sandbox_cards/manifested_card',
   'status': 'already_manifested',
   'action': 'skipped',
   'reason': 'already_manifested'},
  {'card_name': 'private_only',
   'card_path': '/Users/rziiiii/dev/workspace/research/analysis_manifest_batch_backfill_v1/sandbox_cards/private_only',
   'status': 'not_analysis_card',
   'action': 'skipped',
   'reason': 'not_analysis_card'}]}

In [6]:
legacy_manifest = json.loads((legacy / 'analysis_manifest.json').read_text(encoding='utf-8'))
assert batch['backfilled'] == 1
assert legacy_manifest['dataset']['name'] == 'batch_backfill_demo'
assert 'predictions' not in legacy_manifest['paths']
assert 'daily_backtest' not in legacy_manifest['paths']
summary = {
    'schema_version': 'research.analysis_manifest_batch_backfill_v1',
    'status': 'completed',
    'scan_path': str(CARD / 'card_scan.csv'),
    'batch_summary_path': str(CARD / 'batch_backfill_summary.json'),
    'backfilled': batch['backfilled'],
    'skipped': batch['skipped'],
    'manifest_hash': legacy_manifest['manifest_hash'],
    'private_sidecars_hidden': 'predictions' not in legacy_manifest['paths'] and 'daily_backtest' not in legacy_manifest['paths'],
    'conclusion': '批量扫描可先审计旧 analysis card，再只对 backfillable card 生成 analysis_manifest.json。',
}
(CARD / 'analysis_summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
summary

{'schema_version': 'research.analysis_manifest_batch_backfill_v1',
 'status': 'completed',
 'scan_path': '/Users/rziiiii/dev/workspace/research/analysis_manifest_batch_backfill_v1/card_scan.csv',
 'batch_summary_path': '/Users/rziiiii/dev/workspace/research/analysis_manifest_batch_backfill_v1/batch_backfill_summary.json',
 'backfilled': 1,
 'skipped': 2,
 'manifest_hash': 'd240f234b0472748b6a3fc0f22641a7ef595deaf34291881f1738db417e43d93',
 'private_sidecars_hidden': True,
 'conclusion': '批量扫描可先审计旧 analysis card，再只对 backfillable card 生成 analysis_manifest.json。'}

## 中文结论

- 做了什么：新增并验证 analysis card 批量扫描与选择性 manifest 回填入口。
- 核心指标：本 notebook 模拟 3 个目录，识别 1 个 `backfillable`、1 个 `already_manifested`、1 个 `not_analysis_card`，实际只回填 1 个。
- regime 差异：旧 card 中的 `regime_metrics.csv` 被登记进公开 manifest，后续 dashboard/analysis 能继续读取；预测明细和 daily path 不登记。
- 是否晋升：这是框架迁移能力，可晋升到 `src/analysis` 稳定入口；不是策略晋升。
- 风险：默认不自动扫描并改写真实 `research/` 历史产物，避免大规模 churn；批量入口应由 notebook 或 schedule 显式调用。
- 下一步：可以把扫描结果接入 schedule 的 dashboard refresh 前置检查，只提示缺失 manifest 的 card，不默认覆盖。